# Fine-tune Gipformer trên Colab
Chọn runtime GPU. Notebook chuẩn bị môi trường, kiểm rò rỉ rồi chạy pipeline khi người dùng thực thi các cell.
Cấu hình mặc định thử X0 với learning rate 0.0001 và seed 0, rồi so các checkpoint với Gipformer gốc trên `dev_v2_vi`.
Sau khi thêm gói `dev_v2_vi` vào Drive và chọn nhánh chứa cấu hình, đặt `RUN_EXPERIMENT=True` rồi chọn Run all. Notebook kiểm đầu vào và cổng rò rỉ trước khi train.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

BRANCH = "fix/scorer-leak-audit"
EXPERIMENT = "configs/model/experiments/e4_x0_lr1e4_vi.yaml"
SEED = 0
EXPERIMENT_OVERRIDES = []  # Example: ['name=e4_x1_lr1e4', 'train.lr=0.0001']
RUN_EXPERIMENT = False
MIN_FREE_DISK_GB = 5  # Keep space for extraction, manifests, and temporary files.
REPO = Path("/content/gipbench")
ICEFALL = Path("/content/icefall")
DRIVE = Path("/content/drive/MyDrive")
BUNDLE = DRIVE / "gipbench_ft"

if not shutil.which("nvidia-smi"):
    raise RuntimeError("Runtime không có GPU. Chọn Runtime > Change runtime type > GPU (T4 trở lên), rồi chạy lại từ đầu.")
subprocess.run(["nvidia-smi"], check=True)
import torch
print("Python:", sys.version)
print("torch:", torch.__version__, "| CUDA của torch:", torch.version.cuda)
print("CUDA dùng được:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Chọn runtime GPU rồi chạy lại từ đầu.")
print("GPU:", torch.cuda.get_device_name(0))


## Drive và code
Mount Drive; thêm Secret `GH_TOKEN` có quyền đọc repo riêng `vbwork2/Gipbench` và cho notebook truy cập.
Token chỉ truyền trong môi trường tiến trình Git, không nằm trong URL, file cấu hình hay output.
Nếu đã clone, fetch nhánh rồi cập nhật bằng fast-forward; có xung đột thì dừng để xử lý.


In [ ]:
from google.colab import drive, userdata
import base64

drive.mount("/content/drive")
import json
index_path = BUNDLE / 'bundle_index.json'
if not index_path.is_file():
    raise FileNotFoundError(f'Thiếu {index_path}. Chép index mới nhất vào thư mục Drive.')
index = json.loads(index_path.read_text(encoding='utf-8'))
missing_shards = [name for name in sorted(index['shards']) if not (BUNDLE / name).is_file()]
if missing_shards:
    raise FileNotFoundError('Shard còn thiếu trên Drive: ' + ', '.join(missing_shards))
incomplete_shards = [name for name, info in index['shards'].items() if (BUNDLE / name).stat().st_size != info['size']]
if incomplete_shards:
    raise RuntimeError('Shard sai dung lượng trên Drive: ' + ', '.join(sorted(incomplete_shards)))
subprocess.run(["git", "check-ref-format", "--branch", BRANCH], check=True, capture_output=True)

def sync_repo():
    token = userdata.get("GH_TOKEN")
    if not token:
        raise RuntimeError("Thiếu Secret GH_TOKEN hoặc notebook chưa được cấp quyền đọc Secret.")
    auth = base64.b64encode(("x-access-token:" + token).encode()).decode()
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0", GIT_CONFIG_COUNT="1",
               GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
               GIT_CONFIG_VALUE_0="AUTHORIZATION: basic " + auth)

    def git(args, cwd=None, allow_fail=False):
        result = subprocess.run(["git", "-c", "credential.helper=", *args], cwd=cwd,
                                env=env, capture_output=True, text=True)
        if result.returncode and not allow_fail:
            err = (result.stderr or "").replace(token, "***").replace(auth, "***")
            raise RuntimeError("Git thất bại (" + args[0] + "): " + err.strip()[-500:]
                               + "\nKiểm quyền GH_TOKEN, tên nhánh và thay đổi trong clone.")
        return result.returncode

    try:
        url = "https://github.com/vbwork2/Gipbench.git"
        if not (REPO / ".git").is_dir():
            git(["clone", "--depth", "1", "--branch", BRANCH, url, str(REPO)])
        else:
            git(["remote", "set-url", "origin", url], REPO)
            git(["fetch", "origin", BRANCH], REPO)
            exists = git(["show-ref", "--verify", "--quiet", "refs/heads/" + BRANCH], REPO, True) == 0
            git(["checkout", BRANCH] if exists else ["checkout", "-b", BRANCH, "FETCH_HEAD"], REPO)
            git(["merge", "--ff-only", "FETCH_HEAD"], REPO)
    finally:
        env.clear()
        del token, auth

sync_repo()
os.chdir(REPO)
print("Repo:", REPO, "| nhánh:", BRANCH)
subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=REPO, check=True)


## Icefall
Clone nông; ưu tiên commit đã thử `f8c8578c` như `scripts/setup_ft_env.sh`.
Nếu không fetch được commit đó, giữ HEAD hiện có và in commit thực tế; cần kiểm tương thích trước khi train.


In [ ]:
import json
import urllib.request

if not (ICEFALL / ".git").is_dir():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/k2-fsa/icefall", str(ICEFALL)], check=True)
try:
    with urllib.request.urlopen("https://api.github.com/repos/k2-fsa/icefall/commits/f8c8578c", timeout=30) as response:
        tested_commit = json.load(response)["sha"]
    subprocess.run(["git", "fetch", "--depth", "1", "origin", tested_commit], cwd=ICEFALL,
                   check=True, capture_output=True)
    subprocess.run(["git", "checkout", "--detach", tested_commit], cwd=ICEFALL, check=True)
except (OSError, ValueError, KeyError, subprocess.CalledProcessError):
    print("CHƯA ghim được f8c8578c; đang dùng HEAD dưới đây. Kiểm recipe trước khi train.")
subprocess.run(["git", "rev-parse", "HEAD"], cwd=ICEFALL, check=True)


## Chọn k2 theo môi trường đang có
Giữ torch Colab. Đọc [wheel CUDA chính thức](https://k2-fsa.github.io/k2/cuda.html), khớp chính xác torch / CUDA của torch
và tag Python / Linux; CUDA hiển thị bởi nvidia-smi là khả năng của driver, không dùng để chọn wheel.
Không có wheel thì dừng: chọn runtime có bộ Python / torch / CUDA được hỗ trợ, hoặc chủ động cài bộ khớp rồi khởi động lại.
Cell không tự hạ torch hay build k2 từ nguồn.


In [ ]:
from html.parser import HTMLParser
from urllib.parse import unquote, urljoin, urlsplit
from pip._vendor.packaging.tags import sys_tags
from pip._vendor.packaging.utils import parse_wheel_filename
from pip._vendor.packaging.version import Version
import re

K2_INDEX = "https://k2-fsa.github.io/k2/cuda.html"
torch_installed = str(torch.__version__)
torch_version = Version(torch_installed).base_version
cuda_version = torch.version.cuda
if not cuda_version:
    raise RuntimeError("torch không có CUDA. Chọn lại runtime GPU có torch CUDA.")
wanted = f"cuda{cuda_version}.torch{torch_version}"
print("Cần k2:", wanted, "| Python:", sys.version.split()[0])

class WheelLinks(HTMLParser):
    def __init__(self):
        super().__init__()
        self.links = []

    def handle_starttag(self, tag, attrs):
        if tag == "a":
            self.links.extend(value for key, value in attrs if key == "href" and value)

try:
    with urllib.request.urlopen(K2_INDEX, timeout=30) as response:
        page = response.read().decode("utf-8")
except OSError as error:
    raise RuntimeError("Không đọc được danh sách k2; kiểm mạng rồi chạy lại cell.") from error
links = WheelLinks()
links.feed(page)
tags = {tag: i for i, tag in enumerate(sys_tags())}
candidates = []
for href in links.links:
    url = urljoin(K2_INDEX, href)
    filename = unquote(urlsplit(url).path.rsplit("/", 1)[-1])
    if not filename.endswith(".whl"):
        continue
    try:
        name, version, _, wheel_tags = parse_wheel_filename(filename)
    except ValueError:
        continue
    match = re.fullmatch(r"cuda([0-9.]+)\.torch([0-9.]+)", version.local or "")
    compatible = set(tags).intersection(wheel_tags)
    if name == "k2" and match and match.groups() == (cuda_version, torch_version) and compatible:
        candidates.append((version, -min(tags[tag] for tag in compatible), url, filename))
if not candidates:
    raise RuntimeError(
        f"Không có wheel k2 khớp {wanted} và tag Python / Linux hiện tại. "
        f"Xem {K2_INDEX}; đổi runtime hoặc chủ động chọn bộ Python / torch / CUDA có wheel, "
        "khởi động lại runtime rồi chạy từ đầu. Không dùng wheel gần phiên bản.")
_, _, k2_url, k2_filename = max(candidates)
print("Wheel:", k2_filename)


## Cài phụ thuộc và kiểm import
Lọc requirements icefall giống script local (bỏ style check, Cantonese, pypinyin, onnx); cài riêng pypinyin và bộ ONNX
cho stage 5–6. Dùng constraints giữ torch / torchaudio / torchvision hiện có; nếu xung đột thì dừng để xử lý.
Chỉ khi thực thi cell này trên Colab mới cài gói. Kiểm import ở tiến trình Python mới; không chạy train hay decode.


In [ ]:
import importlib.metadata

constraints = Path("/content/gipbench-colab-constraints.txt")
pinned = [f"torch=={torch_installed}"]
for package in ("torchaudio", "torchvision"):
    try:
        pinned.append(f"{package}=={importlib.metadata.version(package)}")
    except importlib.metadata.PackageNotFoundError:
        pass
constraints.write_text("\n".join(pinned) + "\n", encoding="utf-8")
pip_env = dict(os.environ, PIP_CONSTRAINT=str(constraints))

def pip_install(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-c", str(constraints), *args],
                   env=pip_env, check=True)

filtered = []
for line in (ICEFALL / "requirements.txt").read_text(encoding="utf-8").splitlines():
    line = line.strip()
    if line and not line.startswith("#") and not re.search("black|isort|flake8|pycantonese|pypinyin|onnx", line):
        filtered.append(line)
icefall_requirements = Path("/content/icefall-colab-requirements.txt")
icefall_requirements.write_text("\n".join(filtered) + "\n", encoding="utf-8")
pip_install(k2_url)
pip_install("lhotse", "scipy", "pypinyin==0.50.0", "-r", str(icefall_requirements))
pip_install("onnx>=1.15.0", "onnxruntime>=1.16.3", "onnxoptimizer", "onnxsim", "onnxconverter_common")
pip_install("-r", str(REPO / "requirements.txt"))
pip_install("-e", str(REPO))

check_imports = (
    "import torch, k2, lhotse; "
    f"assert str(torch.__version__) == {torch_installed!r}, 'torch đã đổi'; "
    f"assert torch.version.cuda == {cuda_version!r}, 'CUDA của torch đã đổi'; "
    "assert torch.cuda.is_available(), 'CUDA không dùng được'; "
    "print('torch', torch.__version__, 'CUDA', torch.version.cuda); "
    "print('k2', getattr(k2, '__dev_version__', getattr(k2, '__version__', '?'))); "
    "print('lhotse', lhotse.__version__)"
)
subprocess.run([sys.executable, "-c", check_imports], check=True)


## Đường dẫn Colab
Chép cấu hình đã commit; data ở clone, checkpoint / kết quả ở Drive.
Hiện `core.paths.path()` coi tên `python3` là đường dẫn tương đối: đặt ba biến môi trường tới Python tuyệt đối
để các tiến trình con chạy đúng. Đặt lại các biến đường dẫn để cấu hình cũ trong runtime không ghi đè YAML.


In [ ]:
shutil.copyfile(REPO / "configs/paths.colab.yaml", REPO / "configs/paths.yaml")
for key in ("GIPBENCH_DATA", "GIPBENCH_RESULTS", "GIPBENCH_EXP", "ICEFALL"):
    os.environ.pop(key, None)
python3 = shutil.which("python3")
if not python3:
    raise RuntimeError("Không tìm thấy python3 của Colab.")
for key in ("GIPBENCH_PY", "GIPBENCH_FT_PY", "GIPBENCH_ASR_PY"):
    os.environ[key] = python3
for directory in (DRIVE / "gipbench_exp", DRIVE / "gipbench_results"):
    directory.mkdir(parents=True, exist_ok=True)
print("Python chung:", python3)
print((REPO / "configs/paths.yaml").read_text(encoding="utf-8"))


## Giải nén và kiểm release
Đọc shard trực tiếp từ Drive, giải nén vào đĩa local. Cần đủ chỗ cho dữ liệu đã giải nén và phần dự phòng.
Nếu Drive chậm và local đủ chỗ cho cả gói tar, có thể chép nguyên thư mục gói vào `/content/gipbench_ft`, đổi
`BUNDLE` rồi chạy lại cell. Chưa đo phương án nào nhanh hơn trên Colab thật.
`--verify` báo CHƯA CÓ với release chưa đóng gói; stage 0 vẫn phải kiểm đủ đầu vào và mọi bench khoá.


In [ ]:
index = json.loads((BUNDLE / "bundle_index.json").read_text(encoding="utf-8"))
needed = sum(info["size"] for info in index["files"].values())
free = shutil.disk_usage("/content").free
print(f"Local trống: {free / 2**30:.1f} GiB; dữ liệu giải nén: {needed / 2**30:.1f} GiB")
if free < needed + MIN_FREE_DISK_GB * 2**30:
    raise RuntimeError("Thiếu chỗ local để giải nén và giữ phần dự phòng watchdog.")
subprocess.run([python3, "scripts/colab/unpack_ft_bundle.py", "--bundle", str(BUNDLE),
                "--dest", str(REPO)], cwd=REPO, check=True)
subprocess.run([python3, "-m", "gipbench.dataset.release.register", "--verify"], cwd=REPO, check=True)


## Kiểm experiment và dry-run
In các release còn thiếu và lệnh của seed đã chọn. Kết quả dev, checkpoint và artifact nằm trên Drive.


In [ ]:
# Add the editable package to the kernel that was already running before installation.
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))
from gipbench.core import paths, release as R
from gipbench.model.recipe import experiment as X
from gipbench.model.run import Ctx

gate_ok = False
prepared_ok = False
if not (REPO / EXPERIMENT).is_file():
    raise FileNotFoundError(f"Nhánh {BRANCH} chưa chứa {EXPERIMENT}.")
overrides = [*EXPERIMENT_OVERRIDES, "train.watchdog.disk_path=/content", f"train.watchdog.min_free_disk_gb={MIN_FREE_DISK_GB}"]
exp = X.load(REPO / EXPERIMENT, overrides)
if X.catalog_use(exp) != "train" or exp.get("leak_gate", {}).get("mode", "fail") != "fail":
    raise RuntimeError("Colab FT yêu cầu catalog_use: train và leak_gate.mode: fail.")
exp_root = paths.path("exp_root").resolve()
seed = X.run_seeds(exp, SEED if exp.get("seeds") else None, dry=False)[0]
ctx = Ctx(exp, dry=True, seed=seed)
if any(not output.resolve().is_relative_to(exp_root) for output in (ctx.exp_dir, ctx.artifact)):
    raise RuntimeError(f"Checkpoint và artifact phải nằm trong exp_root trên Drive: {exp_root}")
command = [python3, "-m", "gipbench.model.run", EXPERIMENT]
if seed is not None:
    command += ["--seed", str(seed)]
for value in overrides:
    command += ["--set", value]
work = ctx.work
refs = dict.fromkeys([exp["catalog"], *(exp.get("dev") or {}).values(), *X.input_refs(exp), *ctx.guard_refs, *R.locked_bench_refs()])
missing_inputs = [ref for ref in refs if not (R.release_dir(ref) / "release.json").is_file()]
reserved = exp.get("leak_gate", {}).get("reserved_terms")
if reserved and not X.abspath(reserved).is_file():
    missing_inputs.append(reserved)
missing_inputs.extend(path for path in exp.get('leak_gate', {}).get('text_files', []) if not X.abspath(path).is_file())
if missing_inputs:
    print("Đầu vào còn thiếu cho lượt thật:", ", ".join(missing_inputs))

def run_pipeline(stages, dry=False):
    args = [*command, "--stages", stages, *(["--dry-run"] if dry else [])]
    print(" ".join(args), flush=True)
    return subprocess.run(args, cwd=REPO, check=True)

run_pipeline("0-6", dry=True)


## Stage 0 thật: cổng rò rỉ
In `leak_report.md` kể cả khi cổng thoát mã 3. Còn critical chưa miễn trừ thì dừng, sửa dữ liệu rồi chạy lại;
không đổi sang chế độ report để đi tiếp.


In [ ]:
from IPython.display import Markdown, display

gate_ok = False
prepared_ok = False
if not RUN_EXPERIMENT:
    print("Chế độ chuẩn bị. Stage 0 thật đang tắt.")
else:
    if missing_inputs:
        raise RuntimeError("Thiếu đầu vào: " + ", ".join(missing_inputs))
    try:
        run_pipeline("0")
    finally:
        report_md = work / "leak_report.md"
        if report_md.is_file():
            display(Markdown(report_md.read_text(encoding="utf-8")))
    gate = json.loads((work / "leak_report.json").read_text(encoding="utf-8"))
    if gate["blocking"] or not gate["passed"]:
        raise RuntimeError("Cổng rò rỉ chưa đạt.")
    gate_ok = True


## Stage 1–3: curate, Lhotse, trộn cut
Chỉ thực thi sau khi stage 0 đạt. Cell này có thể tốn thời gian / ổ đĩa trên Colab; dữ liệu trung gian ở local.


In [ ]:
prepared_ok = False
if RUN_EXPERIMENT:
    if not gate_ok:
        raise RuntimeError("Phải chạy stage 0 đạt trong phiên này trước.")
    run_pipeline("1-3")
    prepared_ok = True


## Stage 4–6: fine-tune, export / decode dev, chọn artifact
**Thực thi cell này sẽ chạy fine-tune trên GPU.** Watchdog kiểm ổ local `/content`, chừa 5 GiB. Checkpoint nằm ở `exp_root` trên Drive:
mỗi `checkpoint-N.pt` ~1.1 GB, một lượt giữ `segments_per_epoch` cái (~4.4 GB + onnx), nên Drive 15 GB chỉ đủ
1–2 experiment cùng lúc; xoá checkpoint của experiment đã chọn xong artifact trước khi chạy cái tiếp. Chỉnh ngân sách, batch và ngưỡng RAM theo GPU / runtime thực tế.

Mất kết nối nhưng runtime còn sống: kiểm log / tiến trình trước khi chạy lại, tránh hai lượt train cùng ghi checkpoint.
Nếu runtime đã bị thu hồi: chạy lại các cell chuẩn bị, giải nén, stage 0 và stage 1–3 với cùng experiment / seed / mix,
rồi chạy lại stage 4–6. `icefall_runner.resume_args()` tự lấy `checkpoint-N.pt` lớn nhất trên Drive và truyền
`--do-finetune 0 --start-epoch 2 --num-epochs 1 --start-batch N`; không tự thêm cờ resume.
Tiến độ sau checkpoint hoàn chỉnh gần nhất có thể mất. Kiểm checkpoint đọc được nếu Drive ngắt lúc đang ghi;
log / cuts local phải dựng lại, kết quả dev đã có được stage 5 bỏ qua.


In [ ]:
if RUN_EXPERIMENT:
    if not gate_ok or not prepared_ok:
        raise RuntimeError("Phải chạy stage 0 và stage 1–3 đạt trước.")
    # Reserve space for retained epoch checkpoints, one in-progress epoch, and exported models.
    checkpoint_gib = 1.1
    need_drive = (exp["mix"]["epochs"] + exp["train"].get("segments_per_epoch", 4)) * checkpoint_gib + 1.0
    for directory, need in ((Path("/content"), MIN_FREE_DISK_GB), (exp_root, need_drive)):
        free = shutil.disk_usage(directory).free / 2**30
        print(f"{directory}: còn {free:.1f} GiB, cần ~{need:.1f} GiB")
        if free < need:
            raise RuntimeError("Thiếu chỗ trên đĩa local hoặc Drive.")
    run_pipeline("4-6")


## Artifact trên Drive
Liệt kê artifact và model_card của experiment; chưa chạy stage 6 thì chưa có.
Checkpoint ở `output.exp_dir`, kết quả dev ở `gipbench_results`. System YAML sinh trong clone: chép riêng nếu cần giữ.


In [ ]:
artifact = ctx.artifact
print("Checkpoint:", ctx.exp_dir)
print("Kết quả dev:", ctx.rd)
print("Artifact:", artifact)
selection_report = ctx.rd / "select_ckpt.md"
if selection_report.is_file():
    display(Markdown(selection_report.read_text(encoding="utf-8")))
if not artifact.is_dir():
    print("Chưa có artifact; cần stage 6 hoàn tất.")
else:
    for file in sorted(artifact.rglob("*")):
        if file.is_file():
            print(file, f"({file.stat().st_size:,} B)")
    cards = sorted(artifact.rglob("model_card*"))
    print("Model card:", *cards, sep="\n")
